# Libri4Mix Dataset Generator (Kaggle)
This notebook automatically clones the LibriMix repository, installs dependencies, downloads LibriSpeech and WHAM to `/tmp/dataset` (to preserve `/kaggle/working` space), and generates Libri4Mix (n_src=4, 16kHz, max, mix_clean) while staying strictly under the 20 GB Kaggle output limit.

In [ ]:
# Notebook Setup & Installs
!pip install numpy pandas soundfile pyloudnorm pysndfx tqdm
!apt-get update && apt-get install -y sox ffmpeg
!git clone https://github.com/JorisCos/LibriMix.git

In [ ]:
import os
import shutil

storage_dir = "/tmp/dataset"
os.makedirs(storage_dir, exist_ok=True)

def download_extract(url, name, is_zip=False):
    dest = os.path.join(storage_dir, name)
    if not os.path.exists(dest):
        print(f"Downloading {name}...")
        if is_zip:
            !wget -c {url} -O {storage_dir}/{name}.zip
            !unzip -q {storage_dir}/{name}.zip -d {storage_dir}
            !rm {storage_dir}/{name}.zip
        else:
            !wget -c {url} -O {storage_dir}/{name}.tar.gz
            !tar -xzf {storage_dir}/{name}.tar.gz -C {storage_dir}
            !rm {storage_dir}/{name}.tar.gz

download_extract("http://www.openslr.org/resources/12/dev-clean.tar.gz", "dev-clean")
download_extract("http://www.openslr.org/resources/12/test-clean.tar.gz", "test-clean")
download_extract("http://www.openslr.org/resources/12/train-clean-100.tar.gz", "train-clean-100")
download_extract("https://my-bucket-a8b4b49c25c811ee9a7e8bba05fa24c7.s3.amazonaws.com/wham_noise.zip", "wham_noise", is_zip=True)
print("Downloads complete.")

In [ ]:
# Patch pandas deprecation in create_librispeech_metadata.py
patch_file2 = "LibriMix/scripts/create_librispeech_metadata.py"
with open(patch_file2, "r") as f:
    content2 = f.read()
if "error_bad_lines=False" in content2:
    content2 = content2.replace("error_bad_lines=False", "on_bad_lines='skip'")
    with open(patch_file2, "w") as f:
        f.write(content2)
    print("Successfully patched create_librispeech_metadata.py for pandas compatibility.")

# Generate Base Metadata
print("Generating LibriSpeech metadata...")
!python LibriMix/scripts/create_librispeech_metadata.py --librispeech_dir /tmp/dataset/LibriSpeech

print("Generating WHAM noise metadata...")
!python LibriMix/scripts/create_wham_metadata.py --wham_dir /tmp/dataset/wham_noise

In [ ]:
# Patch create_librimix_metadata.py to limit generated dataset size (< 20 GB)
!cd LibriMix && git checkout scripts/create_librimix_metadata.py
patch_file = "LibriMix/scripts/create_librimix_metadata.py"
with open(patch_file, "r") as f:
    content = f.read()

old_code = "        mixtures_md = mixtures_md[:len(mixtures_md) // 100 * 100]\n        mixtures_info = mixtures_info[:len(mixtures_info) // 100 * 100]"
new_code = """\
        if 'train' in librispeech_md_file:
            limit = 2000
        else:
            limit = 200
        mixtures_md = mixtures_md[:limit]
        mixtures_info = mixtures_info[:limit]"""

if old_code in content:
    content = content.replace(old_code, new_code)
    with open(patch_file, "w") as f:
        f.write(content)
    print("Successfully patched create_librimix_metadata.py to limit generated mixtures.")
else:
    print("Warning: Could not patch truncation logic. Size may exceed Kaggle limits.")


In [ ]:
# Generate Libri4Mix Metadata
import os
os.makedirs("Output/Libri4Mix", exist_ok=True)
metadata_out = "Output/Libri4Mix/metadata"
print("Generating Libri4Mix metadata...")
!python LibriMix/scripts/create_librimix_metadata.py \
    --librispeech_dir /tmp/dataset/LibriSpeech \
    --librispeech_md_dir /tmp/dataset/LibriSpeech/metadata \
    --wham_dir /tmp/dataset/wham_noise \
    --wham_md_dir /tmp/dataset/wham_noise/metadata \
    --metadata_outdir Output/Libri4Mix/metadata \
    --n_src 4

In [ ]:
import time
start_time = time.time()
# Generate Libri4Mix Audio
print("Generating Libri4Mix Audio (n_src=4, max mode, 16kHz, mix_clean ONLY)...")
!python LibriMix/scripts/create_librimix_from_metadata.py \
    --librispeech_dir /tmp/dataset/LibriSpeech \
    --wham_dir /tmp/dataset/wham_noise \
    --metadata_dir Output/Libri4Mix/metadata \
    --librimix_outdir Output \
    --n_src 4 \
    --freqs 16k \
    --modes max \
    --types mix_clean

end_time = time.time()
print(f"\nGeneration took {end_time - start_time:.2f} seconds.")

In [ ]:
# Print Final Statistics
import glob
import os

def get_dir_size(path='.'):
    total = 0
    with os.scandir(path) as it:
        for entry in it:
            if entry.is_file():
                total += entry.stat().st_size
            elif entry.is_dir():
                total += get_dir_size(entry.path)
    return total

wav_files = glob.glob("Output/Libri4Mix/wav16k/**/*.wav", recursive=True)
print(f"Total Number of Generated Wav Files: {len(wav_files)}")

output_size_bytes = get_dir_size("Output")
output_size_gb = output_size_bytes / (1024**3)
print(f"Total Output Directory Size: {output_size_gb:.2f} GB")

if output_size_gb > 19.5:
    print("WARNING: Output size is dangerously close to or exceeding the 20 GB limit!")
else:
    print("SUCCESS: Output size is within the 20 GB Kaggle limit.")